In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.gold;

In [0]:
from pyspark.sql.functions import col,avg,min,max,count;


In [0]:
race_results_df = spark.read.table("f1.gold.race_results")

In [0]:
fastest_laps_df = race_results_df.filter(col("fastest_lap_time_ms").isNotNull())

In [0]:
fastest_laps_analysis_df = fastest_laps_df.groupBy(

    "driver_id",
    "driver_name",
    "driver_code",
    "driver_nationality"

).agg(
    count("*").alias("total_fastest_laps_recorded"),
    min("fastest_lap_time_ms").alias("best_fastest_lap_ms"),
    max("fastest_lap_speed").alias("top_speed"),
    avg("fastest_lap_time_ms").alias("avg_fastest_lap_ms"),
    avg("fastest_lap_speed").alias("avg_fastest_lap_speed")
)

In [0]:
fastest_laps_analysis_final_df = fastest_laps_analysis_df.orderBy(
    col("best_fastest_lap_ms").asc()
)

In [0]:
fastest_laps_analysis_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("f1.gold.fastest_laps_analysis")